# 🚀 Titan-GPT V3 Small Model - Colab Training Notebook

This notebook trains the Titan-GPT V3 Small (124M parameters) model on Google Colab with GPU acceleration.

## Features:
- ✅ **GPU Acceleration** (T4/V100/A100)
- ✅ **Google Drive Integration** for checkpoint saving
- ✅ **Error Handling** and recovery
- ✅ **Progress Monitoring** with visualizations
- ✅ **Memory Efficient** training

## Setup Requirements:
1. **Enable GPU**: Runtime → Change runtime type → GPU
2. **Run all cells** in order
3. **Authorize Google Drive** when prompted

## Training Configuration:
- Model: Titan-GPT V3 Small (124M params)
- Context Length: 512 tokens
- Batch Size: 8 (adjustable for GPU)
- Epochs: 10 (customizable)
- Dataset: The Verdict by Edith Wharton

## 📋 Step 1: Check GPU Availability

First, let's verify that GPU is available and check its specifications.

In [ ]:
import torch
import os

# Check GPU availability
print("="*70)
print("GPU STATUS CHECK")
print("="*70)

if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   PyTorch Version: {torch.__version__}")
    device = torch.device("cuda")
else:
    print("⚠️  GPU not available! Training will use CPU (slower).")
    print("   To enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU")
    device = torch.device("cpu")

print(f"\n   Using device: {device}")
print("="*70)

## 💾 Step 2: Mount Google Drive

Mount Google Drive to save training checkpoints and results.

In [ ]:
from google.colab import drive

print("="*70)
print("MOUNTING GOOGLE DRIVE")
print("="*70)

# Mount Google Drive
drive.mount('/content/drive')

# Create directory for checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/Titan_V3_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\n✅ Google Drive mounted successfully")
print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}")
print("="*70)

## 📦 Step 3: Clone Repository and Install Dependencies

Clone the TitanEnhacedGPTS repository and install required packages.

In [ ]:
import subprocess
import sys

print("="*70)
print("CLONING REPOSITORY")
print("="*70)

# Remove existing directory if present
if os.path.exists('/content/TitanEnhacedGPTS'):
    print("Removing existing directory...")
    !rm -rf /content/TitanEnhacedGPTS

# Clone the repository (checking for Colab1 branch)
print("\n📥 Cloning TitanEnhacedGPTS repository...")
try:
    # Try to clone Colab1 branch first
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'Colab1', 
         'https://github.com/Amitsjoysm/TitanEnhacedGPTS.git',
         '/content/TitanEnhacedGPTS'],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        # If Colab1 branch doesn't exist, clone main branch
        print("   Colab1 branch not found, cloning main branch...")
        !git clone --depth 1 https://github.com/Amitsjoysm/TitanEnhacedGPTS.git /content/TitanEnhacedGPTS
    else:
        print("   ✅ Cloned Colab1 branch successfully")
except Exception as e:
    print(f"   Cloning main branch (error with Colab1: {e})")
    !git clone --depth 1 https://github.com/Amitsjoysm/TitanEnhacedGPTS.git /content/TitanEnhacedGPTS

# Change to repository directory
%cd /content/TitanEnhacedGPTS

print("\n✅ Repository cloned successfully")
print("="*70)

In [ ]:
print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

# Install required packages
print("\n📦 Installing required packages...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q tiktoken matplotlib tqdm numpy requests

print("\n✅ Dependencies installed successfully")
print("="*70)

## 🔍 Step 4: Verify Repository Structure

Check that all required files are present.

In [ ]:
print("="*70)
print("VERIFYING REPOSITORY STRUCTURE")
print("="*70)

# Check critical directories and files
required_paths = [
    '/content/TitanEnhacedGPTS/titan-optimal',
    '/content/TitanEnhacedGPTS/titan-optimal/models',
    '/content/TitanEnhacedGPTS/titan-optimal/configs',
    '/content/TitanEnhacedGPTS/ch04/01_main-chapter-code',
]

all_present = True
for path in required_paths:
    if os.path.exists(path):
        print(f"   ✅ {path}")
    else:
        print(f"   ❌ {path} - NOT FOUND")
        all_present = False

if all_present:
    print("\n✅ All required files are present")
else:
    print("\n⚠️  Some files are missing. Training may fail.")

# List titan-optimal contents
print("\n📂 Contents of titan-optimal directory:")
!ls -la /content/TitanEnhacedGPTS/titan-optimal/

print("="*70)

## 🎯 Step 5: Prepare Training Script

Create a comprehensive training function with error handling.

In [ ]:
# Add paths to system
import sys
sys.path.append('/content/TitanEnhacedGPTS')
sys.path.append('/content/TitanEnhacedGPTS/titan-optimal')
sys.path.append('/content/TitanEnhacedGPTS/ch04/01_main-chapter-code')

print("="*70)
print("IMPORTING MODULES")
print("="*70)

try:
    import torch
    import torch.nn as nn
    import tiktoken
    import matplotlib.pyplot as plt
    import time
    import json
    from tqdm import tqdm
    from pathlib import Path
    
    print("✅ Standard modules imported")
    
    # Import custom modules
    from models.titan_gpt_v3 import TitanGPTModelV3
    from configs.model_configs_v3 import get_model_config_v3, get_training_config_v3
    print("✅ Titan V3 modules imported")
    
    # Import dataloader from existing codebase
    from gpt import create_dataloader_v1
    print("✅ Dataloader imported")
    
    print("\n✅ All modules imported successfully")
    
except Exception as e:
    print(f"\n❌ Import error: {e}")
    print("\nTrying to locate files...")
    !find /content/TitanEnhacedGPTS -name "titan_gpt_v3.py" -o -name "model_configs_v3.py" -o -name "gpt.py"

print("="*70)

## 🔧 Step 6: Define Training Functions

Define helper functions for training.

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    """Calculate loss for a single batch."""
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    
    logits, _ = model(input_batch, update_memory=True, mode="train")
    
    loss = nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    """Calculate average loss over data loader."""
    total_loss = 0.0
    
    if len(data_loader) == 0:
        return float("nan")
    
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    model.eval()
    
    with torch.no_grad():
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i >= num_batches:
                break
            
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            logits, _ = model(input_batch, update_memory=False, mode="inference")
            
            loss = nn.functional.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten()
            )
            total_loss += loss.item()
    
    model.train()
    
    return total_loss / num_batches


def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("✅ Helper functions defined")

## 🚀 Step 7: Main Training Function

Define the main training loop with comprehensive monitoring.

In [ ]:
def train_titan_v3_small(
    model_size="small",
    checkpoint_dir="/content/drive/MyDrive/Titan_V3_Checkpoints",
    num_epochs=10,
    batch_size=8,
    save_every_n_epochs=2,
    device="cuda"
):
    """Train Titan-GPT V3 Small model."""
    
    print("\n" + "="*70)
    print("TITAN-GPT V3 SMALL TRAINING")
    print("="*70)
    print(f"Device: {device}")
    print(f"Model Size: {model_size}")
    print(f"Batch Size: {batch_size}")
    print(f"Epochs: {num_epochs}")
    print(f"Checkpoints: {checkpoint_dir}")
    print("="*70)
    
    # Set random seeds
    torch.manual_seed(123)
    if device == "cuda":
        torch.cuda.manual_seed(123)
    
    # Load training data
    print("\n📥 Loading training data...")
    data_path = "/content/TitanEnhacedGPTS/ch05/01_main-chapter-code/the-verdict.txt"
    
    if not os.path.exists(data_path):
        print("   Downloading dataset...")
        import requests
        url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
        response = requests.get(url, timeout=30)
        os.makedirs(os.path.dirname(data_path), exist_ok=True)
        with open(data_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        print("   ✅ Dataset downloaded")
    
    with open(data_path, "r", encoding="utf-8") as f:
        text_data = f.read()
    
    print(f"   ✅ Loaded {len(text_data):,} characters")
    
    # Split data
    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))
    
    # Get configurations
    train_cfg = get_training_config_v3(model_size)
    train_cfg["batch_size"] = batch_size
    train_cfg["num_epochs"] = num_epochs
    
    # Adjust for GPU if needed
    if device == "cuda" and torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        if "T4" in gpu_name:
            train_cfg["batch_size"] = min(batch_size, 8)
        elif "V100" in gpu_name:
            train_cfg["batch_size"] = min(batch_size, 12)
        elif "A100" in gpu_name:
            train_cfg["batch_size"] = min(batch_size, 16)
    
    context_length = 512
    
    # Create dataloaders
    print("\n📊 Creating dataloaders...")
    train_loader = create_dataloader_v1(
        text_data[:split_idx],
        batch_size=train_cfg["batch_size"],
        max_length=context_length,
        stride=context_length,
        drop_last=True,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = create_dataloader_v1(
        text_data[split_idx:],
        batch_size=train_cfg["batch_size"],
        max_length=context_length,
        stride=context_length,
        drop_last=False,
        shuffle=False,
        num_workers=0
    )
    
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    
    # Create model
    print(f"\n🏗️  Creating Titan-GPT V3 {model_size.upper()} model...")
    v3_cfg = get_model_config_v3(model_size, use_memory=True)
    v3_cfg["batch_size"] = train_cfg["batch_size"]
    
    model = TitanGPTModelV3(v3_cfg).to(device)
    
    total_params = count_parameters(model)
    print(f"   Total parameters: {total_params:,}")
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_cfg["learning_rate"],
        weight_decay=train_cfg["weight_decay"]
    )
    
    # Training loop
    print("\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70)
    
    train_losses = []
    val_losses = []
    perplexities = []
    best_val_loss = float('inf')
    
    model.train()
    
    for epoch in range(num_epochs):
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{num_epochs}")
        print(f"{'='*70}")
        
        epoch_start = time.time()
        epoch_loss = 0.0
        
        # Training loop with progress bar
        pbar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
        for batch_idx, (input_batch, target_batch) in enumerate(pbar):
            # Forward pass
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            
            # Backward
            loss = loss / train_cfg["gradient_accumulation_steps"]
            loss.backward()
            
            epoch_loss += loss.item()
            
            # Update weights
            if (batch_idx + 1) % train_cfg["gradient_accumulation_steps"] == 0:
                optimizer.step()
                optimizer.zero_grad()
            
            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Validation
        val_loss = calc_loss_loader(val_loader, model, device)
        train_loss = epoch_loss / len(train_loader)
        perplexity = torch.exp(torch.tensor(val_loss)).item()
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        perplexities.append(perplexity)
        
        epoch_time = time.time() - epoch_start
        
        print(f"\n📊 Epoch {epoch+1} Summary:")
        print(f"   Train Loss: {train_loss:.4f}")
        print(f"   Val Loss: {val_loss:.4f}")
        print(f"   Perplexity: {perplexity:.2f}")
        print(f"   Time: {epoch_time:.2f}s")
        
        # Save checkpoint
        if (epoch + 1) % save_every_n_epochs == 0 or val_loss < best_val_loss:
            checkpoint_path = os.path.join(
                checkpoint_dir,
                f"titan_v3_{model_size}_epoch_{epoch+1}.pth"
            )
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'perplexity': perplexity,
                'config': v3_cfg,
            }, checkpoint_path)
            print(f"   💾 Checkpoint saved: {os.path.basename(checkpoint_path)}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_checkpoint_path = os.path.join(
                    checkpoint_dir,
                    f"titan_v3_{model_size}_best.pth"
                )
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'perplexity': perplexity,
                    'config': v3_cfg,
                }, best_checkpoint_path)
                print(f"   ⭐ Best model saved: {os.path.basename(best_checkpoint_path)}")
        
        # Reset short-term memory
        if hasattr(model, 'reset_memory'):
            model.reset_memory(level="short")
    
    # Save final results
    results = {
        "model_size": model_size,
        "total_params": total_params,
        "num_epochs": num_epochs,
        "batch_size": train_cfg["batch_size"],
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "final_perplexity": perplexities[-1],
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "perplexities": perplexities,
    }
    
    results_path = os.path.join(checkpoint_dir, f"training_results_{model_size}.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    print(f"Model: Titan-GPT V3 {model_size.upper()}")
    print(f"Parameters: {total_params:,}")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print(f"Final Perplexity: {perplexities[-1]:.2f}")
    print(f"\n📁 All files saved to: {checkpoint_dir}")
    print("="*70)
    
    return results


print("✅ Training function defined")

## 🎯 Step 8: Configure and Run Training

Set training parameters and start training.

In [ ]:
# Training Configuration
MODEL_SIZE = "small"          # Options: "small" (124M params)
NUM_EPOCHS = 10               # Number of training epochs
BATCH_SIZE = 8                # Batch size (will auto-adjust for GPU)
SAVE_EVERY_N_EPOCHS = 2       # Save checkpoint frequency

# Determine device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*70)
print("TRAINING CONFIGURATION")
print("="*70)
print(f"Model Size: {MODEL_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Device: {DEVICE}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print("="*70)

# Run training
try:
    results = train_titan_v3_small(
        model_size=MODEL_SIZE,
        checkpoint_dir=CHECKPOINT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        save_every_n_epochs=SAVE_EVERY_N_EPOCHS,
        device=DEVICE
    )
    
    print("\n🎉 Training completed successfully!")
    
except Exception as e:
    print(f"\n❌ Training failed with error: {e}")
    import traceback
    traceback.print_exc()

## 📊 Step 9: Visualize Training Results

Create plots of training progress.

In [ ]:
import matplotlib.pyplot as plt

# Plot training progress
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
axes[0].plot(range(1, len(results['train_losses'])+1), results['train_losses'], 
             marker='o', label='Train Loss', color='blue', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Validation Loss
axes[1].plot(range(1, len(results['val_losses'])+1), results['val_losses'], 
             marker='o', label='Val Loss', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Validation Loss', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Perplexity
axes[2].plot(range(1, len(results['perplexities'])+1), results['perplexities'], 
             marker='o', label='Perplexity', color='green', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Perplexity', fontsize=12)
axes[2].set_title('Perplexity (Lower is Better)', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot to Google Drive
plot_path = f"{CHECKPOINT_DIR}/training_progress_{MODEL_SIZE}.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Plot saved to: {plot_path}")

plt.show()

# Print summary statistics
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)
print(f"Model: Titan-GPT V3 {MODEL_SIZE.upper()}")
print(f"Total Parameters: {results['total_params']:,}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"\nFinal Metrics:")
print(f"  Train Loss: {results['final_train_loss']:.4f}")
print(f"  Val Loss: {results['final_val_loss']:.4f}")
print(f"  Best Val Loss: {results['best_val_loss']:.4f}")
print(f"  Final Perplexity: {results['final_perplexity']:.2f}")
print(f"\nImprovement:")
print(f"  Train Loss: {results['train_losses'][0]:.4f} → {results['final_train_loss']:.4f} "
      f"({((results['train_losses'][0] - results['final_train_loss']) / results['train_losses'][0] * 100):.1f}% reduction)")
print(f"  Val Loss: {results['val_losses'][0]:.4f} → {results['final_val_loss']:.4f} "
      f"({((results['val_losses'][0] - results['final_val_loss']) / results['val_losses'][0] * 100):.1f}% reduction)")
print("="*70)

## 💾 Step 10: List Saved Checkpoints

View all saved files in Google Drive.

In [ ]:
print("="*70)
print("SAVED FILES IN GOOGLE DRIVE")
print("="*70)

# List all files in checkpoint directory
files = sorted(os.listdir(CHECKPOINT_DIR))

if files:
    print(f"\n📁 Files in {CHECKPOINT_DIR}:\n")
    
    total_size = 0
    for file in files:
        file_path = os.path.join(CHECKPOINT_DIR, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        total_size += file_size
        
        # Different icons for different file types
        if file.endswith('.pth'):
            icon = "🔹"
        elif file.endswith('.json'):
            icon = "📄"
        elif file.endswith('.png'):
            icon = "📊"
        else:
            icon = "📎"
        
        print(f"   {icon} {file:<50} {file_size:>8.2f} MB")
    
    print(f"\n   Total size: {total_size:.2f} MB")
else:
    print("   No files found.")

print("\n" + "="*70)
print("\n✅ All files are accessible in:")
print("   Google Drive → MyDrive → Titan_V3_Checkpoints")
print("="*70)

## 🔍 Step 11: Load and Inspect Best Model (Optional)

Load the best checkpoint and inspect its details.

In [ ]:
# Load best checkpoint
best_checkpoint_path = f"{CHECKPOINT_DIR}/titan_v3_{MODEL_SIZE}_best.pth"

if os.path.exists(best_checkpoint_path):
    print("="*70)
    print("LOADING BEST MODEL")
    print("="*70)
    
    checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE)
    
    print(f"\n📥 Loaded checkpoint from: {os.path.basename(best_checkpoint_path)}")
    print(f"\n📊 Checkpoint Information:")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Train Loss: {checkpoint['train_loss']:.4f}")
    print(f"   Val Loss: {checkpoint['val_loss']:.4f}")
    print(f"   Perplexity: {checkpoint['perplexity']:.2f}")
    
    # Model configuration
    if 'config' in checkpoint:
        cfg = checkpoint['config']
        print(f"\n⚙️  Model Configuration:")
        print(f"   Embedding dim: {cfg['emb_dim']}")
        print(f"   Num layers: {cfg['n_layers']}")
        print(f"   Num heads: {cfg['n_heads']}")
        print(f"   Context length: {cfg['context_length']}")
        print(f"   Memory variant: {cfg.get('memory_variant', 'N/A')}")
    
    print("\n✅ Best model is ready for inference or fine-tuning!")
    print("="*70)
else:
    print(f"⚠️  Best checkpoint not found at: {best_checkpoint_path}")

## 🎉 Training Complete!

### What's Been Saved:
1. **Model Checkpoints** (`.pth` files):
   - Best performing model: `titan_v3_small_best.pth`
   - Epoch checkpoints: `titan_v3_small_epoch_N.pth`

2. **Training Metrics** (`.json` file):
   - Complete training history
   - Loss curves and perplexity

3. **Visualizations** (`.png` file):
   - Training progress plots

### Next Steps:
1. **Download Checkpoints**: Access files from Google Drive
2. **Use for Inference**: Load best model for text generation
3. **Fine-tune**: Continue training on custom datasets
4. **Experiment**: Try different hyperparameters

### Tips:
- For longer training: Use Colab Pro for extended runtime
- For better performance: Try larger batch sizes on better GPUs
- Monitor GPU: Run `!nvidia-smi` to check utilization
- Save frequently: Reduce `save_every_n_epochs` for long sessions

### Resources:
- [Build a Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch)
- [Titan Architecture Paper](https://arxiv.org/abs/2501.00663)
- [Google Colab Documentation](https://colab.research.google.com/notebooks/intro.ipynb)

---

**Happy Training! 🚀**

## 🔧 Troubleshooting & Utilities

In [ ]:
# Check GPU memory usage
!nvidia-smi

In [ ]:
# Check disk space in Google Drive
!df -h /content/drive

In [ ]:
# Clear GPU memory cache (if needed)
import torch
torch.cuda.empty_cache()
print("✅ GPU memory cache cleared")